In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nrgpt_analysis import (load_nrgpt, load_gpt2, make_context, per_word_energies, per_word_surprisal,
                             per_word_surprisal_gpt2,
                             per_word_predictive_energies,
                             per_word_predictive_conditional_energies,
                             plot_energy_landscape_2d, plot_energy_landscape_pca)

MODEL = "nrgpt_local"
MODEL = "../nrgpt/out-OWT02_owt_best_configs/Best_OWT02_owt_best_configs_model=NRGPT_H_FF2W_embed=1536_depth=6_heads=12_LR=3e-05_minLR=None_minLrDiv=10.0_numIter=100000_exp_kko52p3j.pt"

model, tokenizer = load_nrgpt(MODEL)
ctx = make_context(model, tokenizer)
print(f"Block: {ctx.block.__class__.__name__}  |  block_size={ctx.block_size}")

Model loaded from ../nrgpt/out-OWT02_owt_best_configs/Best_OWT02_owt_best_configs_model=NRGPT_H_FF2W_embed=1536_depth=6_heads=12_LR=3e-05_minLR=None_minLrDiv=10.0_numIter=100000_exp_kko52p3j.pt
Block: BlockGrad_FF2W  |  block_size=1024


In [2]:
# getting gpt2 small

modelgpt2, tokenizergpt2 = load_gpt2("gpt2")


In [4]:
enc = tokenizergpt2("In recent years, researchers have discovered that ", return_tensors="pt")
with torch.no_grad():
      output = modelgpt2.generate(
      enc.input_ids,
      attention_mask=enc.attention_mask,
      pad_token_id=tokenizergpt2.eos_token_id,
      max_new_tokens=20,
  )
print(tokenizergpt2.decode(output[0]))

In recent years, researchers have discovered that erythrocytes are the most abundant of all living cells in the body. They are the most


In [5]:
stories = pd.read_csv("psych_data/nsc_data/all_stories.tok", sep="\t")
rts = pd.read_csv("psych_data/nsc_data/processed_RTs.tsv", sep="\t")

print("Stories shape:", stories.shape, "| items:", sorted(stories['item'].unique()))
print("RTs shape:", rts.shape)

# Per-word mean RT across subjects (filter to correct trials only).
rt_clean = rts[rts['correct'] >= 5]  # comprehension question correctness threshold
word_rt = (rt_clean.groupby(['item', 'zone'])
                   .agg(mean_RT=('RT', 'mean'), n_subj=('RT', 'count'))
                   .reset_index())
print(f"\nUnique (item, zone) cells: {len(word_rt)}")
print(word_rt.head())

Stories shape: (10256, 3) | items: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
RTs shape: (848857, 12)

Unique (item, zone) cells: 10256
   item  zone     mean_RT  n_subj
0     1     1  369.011905      84
1     1     2  368.183908      87
2     1     3  344.318182      88
3     1     4  354.639535      86
4     1     5  349.674157      89


In [6]:
all_word_rows = []
for item in sorted(stories['item'].unique()):
    words = stories[stories['item'] == item].sort_values('zone')['word'].tolist()
    records_c, _      = per_word_predictive_conditional_energies(words, ctx)
    records_e, ranges = per_word_energies(words, ctx)
    surprisals, _     = per_word_surprisal(words, ctx, n_steps=6)
    surprisalsgpt2, _     = per_word_surprisal_gpt2(words, modelgpt2, tokenizergpt2)
    records_p, _      = per_word_predictive_energies(words, ctx)
    n_tokens = ranges[-1][1] if ranges else 0
    print(f"item {item}: {len(words)} words, {len(records_e)} processed, {n_tokens} BPE tokens")
    for i, (rec_e, surp, surpgpt2, rec_p, rec_c) in enumerate(zip(records_e, surprisals, surprisalsgpt2, records_p, records_c)):
        all_word_rows.append({"item": item, "zone": i + 1,
                              **rec_e, **rec_p, **rec_c,
                              "surprisal": surp,
                              "surprisalgpt2": surpgpt2})

energies = pd.DataFrame(all_word_rows)
merged = energies.merge(word_rt, on=["item", "zone"], how="inner")
print(f"\nmerged shape: {merged.shape}")
print(merged.head())

item 1: 1073 words, 850 processed, 1024 BPE tokens
item 2: 990 words, 871 processed, 1024 BPE tokens
item 3: 1040 words, 874 processed, 1024 BPE tokens
item 4: 1085 words, 867 processed, 1023 BPE tokens
item 5: 1013 words, 869 processed, 1024 BPE tokens
item 6: 1099 words, 840 processed, 1024 BPE tokens
item 7: 999 words, 882 processed, 1024 BPE tokens
item 8: 980 words, 840 processed, 1021 BPE tokens
item 9: 1038 words, 805 processed, 1024 BPE tokens
item 10: 939 words, 801 processed, 1024 BPE tokens

merged shape: (8499, 142)
   item  zone     word  n_bpe  start     E_attn_0      E_ff_0    E_total_0  \
0     1     1       If      1      0  -165.803833 -357.604645  -523.408447   
1     1     2      you      1      1 -1453.765503 -714.486816 -2168.252441   
2     1     3     were      1      2 -1602.756104 -763.945618 -2366.701660   
3     1     4       to      1      3 -2729.261475 -122.555954 -2851.817383   
4     1     5  journey      1      4 -1453.008911 -914.317078 -2367.325928  

In [7]:
merged_full = energies.merge(rt_clean, on=["item", "zone"], how="inner")
print(f"merged_full shape: {merged_full.shape}")
print(merged_full.head())

merged_full shape: (704812, 150)
   item  zone word_x  n_bpe  start    E_attn_0      E_ff_0   E_total_0  \
0     1     1     If      1      0 -165.803833 -357.604645 -523.408447   
1     1     1     If      1      0 -165.803833 -357.604645 -523.408447   
2     1     1     If      1      0 -165.803833 -357.604645 -523.408447   
3     1     1     If      1      0 -165.803833 -357.604645 -523.408447   
4     1     1     If      1      0 -165.803833 -357.604645 -523.408447   

    E_attn_1      E_ff_1  ...        WorkerId  WorkTimeInSeconds  correct  \
0  899.55542  754.273438  ...  A3QJPB0NZU5PY1               3960        6   
1  899.55542  754.273438  ...  A2RPQGUWVZPX7U               2431        5   
2  899.55542  754.273438  ...  A11KMPAZSE5Q0Q               1287        5   
3  899.55542  754.273438  ...  A1U1QL617G5DU3               2074        6   
4  899.55542  754.273438  ...   ACTW5YEWV9OR0               2213        6   

    RT  word_y  nItem  meanItemRT    sdItemRT  gmeanItemRT 

In [8]:
# Unigram and bigram counts (one row per item/zone via the ".whole" entries).
freq1 = pd.read_csv("psych_data/nsc_data/freqs-1.tsv", sep="\t", header=None,
                    names=["fid", "type", "word_f", "unigram", "_u_ctx"])
freq2 = pd.read_csv("psych_data/nsc_data/freqs-2.tsv", sep="\t", header=None,
                    names=["fid", "type", "word_f", "bigram", "_b_ctx"])
freq1 = freq1[freq1["fid"].str.endswith(".whole")].copy()
freq2 = freq2[freq2["fid"].str.endswith(".whole")].copy()
freq1["item"] = freq1["fid"].str.split(".").str[0].astype(int)
freq2["item"] = freq2["fid"].str.split(".").str[0].astype(int)
freq1["zone"] = freq1["fid"].str.split(".").str[1].astype(int)
freq2["zone"] = freq2["fid"].str.split(".").str[1].astype(int)
freq1["log_unigram"] = np.log(pd.to_numeric(freq1["unigram"], errors="coerce").clip(lower=1))
freq2["log_bigram"] = np.log(pd.to_numeric(freq2["bigram"], errors="coerce").clip(lower=1))
freqs = freq1[["item", "zone", "log_unigram"]].merge(
    freq2[["item", "zone", "log_bigram"]], on=["item", "zone"], how="left")

# Assemble model frame.
df = merged_full.merge(freqs, on=["item", "zone"], how="left").copy()
df["word_length"] = df["word_x"].astype(str).str.len() if "word_x" in df.columns else df["word"].astype(str).str.len()








In [9]:
freqs

,item,zone,log_unigram,log_bigram
0,1,1,18.628843,18.417000
1,1,2,20.175287,17.044143
2,1,3,19.941297,15.863419
3,1,4,22.163889,15.913385
4,1,5,15.736359,11.041625
...,...,...,...,...
10251,10,935,22.306978,13.795200
10252,10,936,17.707253,14.012504
10253,10,937,14.981096,8.354910
10254,10,938,21.051339,12.391298


In [10]:
print(df)

        item  zone word_x  n_bpe  start     E_attn_0      E_ff_0    E_total_0  \
0          1     1     If      1      0  -165.803833 -357.604645  -523.408447   
1          1     1     If      1      0  -165.803833 -357.604645  -523.408447   
2          1     1     If      1      0  -165.803833 -357.604645  -523.408447   
3          1     1     If      1      0  -165.803833 -357.604645  -523.408447   
4          1     1     If      1      0  -165.803833 -357.604645  -523.408447   
...      ...   ...    ...    ...    ...          ...         ...          ...   
704807    10   801   many      1   1023 -4172.869629 -351.368958 -4524.238770   
704808    10   801   many      1   1023 -4172.869629 -351.368958 -4524.238770   
704809    10   801   many      1   1023 -4172.869629 -351.368958 -4524.238770   
704810    10   801   many      1   1023 -4172.869629 -351.368958 -4524.238770   
704811    10   801   many      1   1023 -4172.869629 -351.368958 -4524.238770   

           E_attn_1      E_

In [ ]:
df.to_csv("nsc_with_energy_surprisal_measures_100.csv", index=False)

In [ ]:
import statsmodels.formula.api as smf

df["log_RT"] = np.log(df["RT"].astype(float))

# Center continuous predictors for cleaner mixed-model fits.
predictors = ["word_length", "zone", "log_unigram", "log_bigram", "n_bpe",
              "E_attn_6", "E_ff_6", "E_total_6", "surprisal",
              "E_attn_pred_sum_5", "E_ff_pred_sum_5", "E_total_pred_sum_5",
              "E_attn_token_sum_5", "E_ff_token_sum_5", "E_total_token_sum_5"]
for col in predictors:
    df[col + "_c"] = df[col] - df[col].mean()

df_fit = df.dropna(subset=["log_RT"] + [c + "_c" for c in predictors]).copy()
df_fit["_grp"] = 1  # statsmodels needs a top-level group; subject + item enter as variance components.

covariates = "word_length_c + zone_c + log_unigram_c + log_bigram_c + n_bpe_c"
effects    = ("E_attn_6_c + E_ff_6_c + E_total_6_c + surprisal_c "
              "+ E_attn_pred_sum_5_c + E_ff_pred_sum_5_c + E_total_pred_sum_5_c "
              "+ E_attn_token_sum_5_c + E_ff_token_sum_5_c + E_total_token_sum_5_c")

# Crossed random intercepts for subject (WorkerId) and item via vc_formula.
md = smf.mixedlm(f"log_RT ~ {covariates} + {effects}",
                 data=df_fit,
                 groups=df_fit["_grp"],
                 re_formula="0",
                 vc_formula={"subject": "0 + C(WorkerId)",
                             "item":    "0 + C(item)"})
res = md.fit(method="lbfgs")
print(res.summary())

# Baseline (covariates only), useful for a likelihood-ratio comparison.
md0 = smf.mixedlm(f"log_RT ~ {covariates}",
                  data=df_fit,
                  groups=df_fit["_grp"],
                  re_formula="0",
                  vc_formula={"subject": "0 + C(WorkerId)",
                              "item":    "0 + C(item)"})
res0 = md0.fit(method="lbfgs")
print(f"\nbaseline logLik={res0.llf:.1f}  |  full logLik={res.llf:.1f}  |  diff={res.llf - res0.llf:.1f}")